![](https://ga-dash.s3.amazonaws.com/production/assets/logo-9f88ae6c9c3871690e33280fcf557f33.png) 
# SQL IV: Introduction to SQlite and CRUD - PetDB Create and CRUD

_Author:_ Andres Gonzalez Flores

In this activity, you will learn how to use SQLite3 and python to create a file-based database and populate it with pets. For this you will explore how the CRUD paradigm works within the context of a database and SQL commands. 

Learning Objectives:

- Explain the concept of CRUD operations (Create, Read, Update, Delete) within the context of database management and illustrate how each operation relates to specific SQL commands (`INSERT`, `SELECT`, `UPDATE`, `DELETE`)
- Generate a quick, file-based SQL database with SQLite3
- Carry out table creation, row insertion (C), entry manipulation (U), and entry deletion (D) using SQL commands.


In [1]:
# Importing libraries
import sqlite3
import pandas as pd
#if you dont have the libraries installed, you can install them using pip (just uncomment the following lines)
# %pip install pandas
# %pip install sqlite3

When using sqlite, the first thing we need to do is to connect to the database. Thanks for the sqlite3 library, this is very easy to do:

# Creating your own database following the CRUD paradigm
Remember CRUD stands for: 
- **C**reate: add new data to the database
- **R**ead: retrieve data from the database
- **U**pdate: update existing data in the database
- **D**elete: remove data from the database


The first step you need to take when interacting with a SQlite DB is connecting to your SQLite DB file. We need to do this before we get started.

In [81]:
conn = sqlite3.connect('pet.db') # this connect to your SQLite database, if it doesn't exist, it will create it for you.
cur = conn.cursor() # this creates a cursor object that will allow us to execute SQL commands


## 1. **C**reate

Now that you have your database connection and cursor, you can start interacting with your database. Following the CRUD paradigm, you can start with the letter 'C' for create. 

Imagine you're making a spreadsheet to keep track of your friends' pets. SQLite lets you do something similar, but instead of a spreadsheet, you create a "table."

To create a table, you need to use the `CREATE TABLE` command. This command will create a new table with the specified columns and their data types.

| Field | Type |
|-------|------|
| pet_name | str |
| pet_age | int |
| pet_species | str |
| pet_owner | str |


In [67]:
# First attempt to drop the table if it exists
try:
    cur.execute('DROP TABLE pets')
except sqlite3.OperationalError:
    pass  # Table doesn't exist yet, that's okay!

# Now create the table
cur.execute("""
CREATE TABLE pets (
    pet_name TEXT PRIMARY KEY NOT NULL,
    pet_age INTEGER,
    pet_species TEXT NOT NULL,
    pet_owner TEXT
)
""")

conn.commit()  # Save the changes

Let's break down the CREATE TABLE commands:

- cur.execute(...): This line tells SQLite to execute the command within the parentheses. cur is a cursor object, which is like a messenger that carries your commands to the database.

- CREATE TABLE pets: This is the core of the command. It tells SQLite to create a new table and give it the name "pets". If a table with that name already exists, you'd typically use DROP TABLE pets beforehand to avoid an error.

| Column Name | Data Type | Constraints | Description |
|---|---|---|---|
| `pet_name` | `TEXT` | `PRIMARY KEY`, `NOT NULL` | The pet's name. Must be unique and cannot be left blank. |
| `pet_age` | `INTEGER` |  | The pet's age (a whole number). Can be left blank. |
| `pet_species` | `TEXT` | `NOT NULL` | The type of pet (e.g., "dog", "cat"). Cannot be left blank. |
| `pet_owner` | `TEXT` |  | The pet owner's name. Can be left blank. |

*Note: The `PRIMARY KEY` constraint ensures that the `pet_name` column is unique and cannot be left blank. The `NOT NULL` constraint ensures that every entry in the `pet_name` column must have a value. The `INTEGER` data type stores whole numbers, and the `TEXT` data type stores text strings.

Now that the table is created, you can insert data using the `INSERT INTO` command, which adds a new row with the specified values.

To simplify this, you can create a function called `insert_pet`. This function will take the cursor, name, age, species, and human as arguments and insert the corresponding data into the table, making it easier to add new pets.

In [68]:
#create the insert_pet function
def insert_pet(cur, pet_name, pet_age, pet_species, pet_owner):
    insert_query = f"""
    INSERT INTO pets VALUES ("{pet_name}", {pet_age}, "{pet_species}", "{pet_owner}")
    """
    
    cur.execute(insert_query)


Now we can use the insert_pet function to insert data into the table. For example, if you want to insert data about your imaginary friend's Tim, Mridul, Cesar and Cameron's pets, you can do the following:   

In [69]:
# C for Create
insert_pet(cur, "Ella", 9, "cat", "Tim")
insert_pet(cur, "Max", 12, "dog", "Mridul")
insert_pet(cur, "Rainbow", 1, "fish", "Mridul")
insert_pet(cur, "Sage", 8, "cat", "Cameron")
insert_pet(cur, "Cocoa", 8, "cat", "Cameron")
insert_pet(cur, "Spots", 3, "dog", "Cesar")

# 2. **R**ead
Now that you have inserted the data, you can **R**ead the data from the table using the SQL `SELECT` command. Again to make things easier, you can create a function called `read_all_pets` that will return a pandas DataFrame with all the rows from your pets table.

In [70]:
# R for Read
def read_all_pets(cur):
    res = cur.execute("SELECT * FROM pets;")
    return pd.DataFrame(res.fetchall(), columns=['pet_name', 'pet_age', 'pet_species', 'pet_owner'])

Lets break down the `read_all_pets` function:
- `cur.execute("SELECT * FROM pets;")`: this executes the SQL command to select all the rows from the pets table.
- `res.fetchall()`: this fetches all the rows from the pets table.
- `pd.DataFrame(res.fetchall(), columns=['pet_name', 'pet_age', 'pet_species', 'pet_owner'])`: this creates a pandas DataFrame with the rows from the pets table and the columns pet_name, pet_age, pet_species, and pet_owner.


In [71]:
pets_df = read_all_pets(cur)
pets_df


,name,age,species,human
0,Ella,9,cat,Tim
1,Max,12,dog,Mridul
2,Rainbow,1,fish,Mridul
3,Sage,8,cat,Cameron
4,Cocoa,8,cat,Cameron
5,Spots,3,dog,Cesar



# 3. **U**pdate
Now, say you want to update the age of a pet every time they have a birthday. You can use the `UPDATE` (**U** for Update) command to do this. To make things easier, you can create a function called `pet_birthday` that will update the age of a pet by 1.

In [72]:
# U for Update
def pet_birthday(cur, pet_name):
    birthday_query = f"""
    UPDATE pets 
    SET pet_age = pet_age + 1 
    WHERE pet_name = "{pet_name}" 
    """

    cur.execute(birthday_query)

Lets break down the UPDATE command:
- `UPDATE` pets: this specifies the table we want to update
- `SET` age = age + 1: this specifies the new value we want to set
- `WHERE` name = "{name}": this specifies the condition that tells us which row to update

Now, every time you call the `pet_birthday` function, it will update the age of the pet by 1. Let's see how it works:

In [73]:
pet_birthday(cur, 'Ella') #every time you call the function, the age of the pet Ella will increase by 1

read_all_pets(cur)

,name,age,species,human
0,Ella,10,cat,Tim
1,Max,12,dog,Mridul
2,Rainbow,1,fish,Mridul
3,Sage,8,cat,Cameron
4,Cocoa,8,cat,Cameron
5,Spots,3,dog,Cesar



# 4. **D**elete
If you need to remove an entry from your database, you can use the `DELETE` (**D** for Delete) command. To make things easier, you can create a function called `drop_pet` that will delete a pet from the table.

In [76]:
# D for Delete
def drop_pet(cur, pet_name):
    del_query = f"""
    DELETE FROM pets
    WHERE pet_name = "{pet_name}"
    """
    
    cur.execute(del_query)

Lets break down the DELETE command:     
- `DELETE FROM pets`: this specifies the table we want to delete from
- `WHERE name = "{name}"`: this specifies the condition that tells us which row to delete

In [77]:
drop_pet(cur, 'Ella')

By running the `drop_pet` function, the pet Ella will be removed from the table. We can see that the pet Ella is no longer in the table by running the `read_all_pets` function. We can also see that the pet Ella is no longer in the table by directly running a SQL query using the `SELECT` command.


In [78]:
read_all_pets(cur)

,name,age,species,human
0,Max,12,dog,Mridul
1,Rainbow,1,fish,Mridul
2,Sage,8,cat,Cameron
3,Cocoa,8,cat,Cameron
4,Spots,3,dog,Cesar


In [61]:
cur.execute("SELECT * FROM pets").fetchall()

[('Max', 12, 'dog', 'Mridul'),
 ('Rainbow', 2, 'fish', 'Mridul'),
 ('Sage', 8, 'cat', 'Cameron'),
 ('Cocoa', 8, 'cat', 'Cameron'),
 ('Spots', 3, 'dog', 'Cesar')]

After you are done with your database, you need to close the connection to the database. This is done by calling the `close` method on the connection object.

In [ ]:
# close your connection to the database
conn.close()

## Recap

Nice! you have now sucesfully created a SQLite database and executed at least one operation under each of the steps of the CRUD paradigm. To recap: 

**CRUD:**
- **C**reate: to add new data to your database you used     the `INSERT INTO` SQL command.
- **R**ead: to retrieve data from your database you used the `SELECT` SQL command.
- **U**pdate: to update existing data in your database you used the `UPDATE` SQL command.
- **D**elete: remove data from the database. For this we used the `DELETE` SQL command.

**Working with SQLite in python:**
To work with SQLIte in python you need to follow these steps:
1. Connect to the database
2. Perform CRUD operations, depending on your needs, such as:
   1. Create the table
   2. Insert the data
   3. Read the data
   4. Update the data
   5. Delete the data
3. Close the connection to the database

